# ALiBi — Attention with Linear Biases

源码导航：[core/position/alibi.py](../../../core/position/alibi.py) 中的 `ALiBiPositionalBias` 与 `get_alibi_slopes`。

Press et al. (2021) 在 *Train Short, Test Long: Attention with Linear Biases Enables Input Length Extrapolation* 中提出 ALiBi。与所有在 embedding 层注入位置信息的方法（Learned/Sinusoidal/RoPE）不同，ALiBi **直接在注意力分数上施加负的线性偏置**，使模型天然具备局部性偏好，并能零成本外推到任意长度。

### 1. 理论推导

#### 1.1 核心公式

对于注意力头 $h$，ALiBi 将标准注意力分数修改为：

$$\text{score}'_h(i, j) = \text{score}_h(i, j) - m_h \cdot |i - j|$$

其中：
- $m_h > 0$ 是头 $h$ 的固定斜率；
- $|i - j|$ 是 query 位置 $i$ 与 key 位置 $j$ 的绝对距离。

**效果**：距离越远的 token 对，其注意力分数被惩罚得越重，形成强烈的**局部性先验**。

#### 1.2 斜率生成

ALiBi 的斜率是一个几何递减序列：

$$m_h = 2^{-8 \cdot h / H}, \quad h = 1, 2, \ldots, H$$

其中 $H$ 为头数。最小头（$h=1$）的斜率最大，对远距离惩罚最强；最大头（$h=H$）的斜率最小，允许更宽的注意力范围。不同头因此学习到不同尺度的位置依赖。

#### 1.3 与 RoPE/绝对编码的本质区别

| 维度 | ALiBi | RoPE | 绝对编码 |
|---|---|---|---|
| 注入层级 | Attention score | Q/K 向量 | Input embedding |
| 外推性 | 零成本（无需修改） | 需 NTK/YaRN 调整 | 无法外推 |
| 局部性先验 | 显式（线性惩罚） | 隐式（旋转周期） | 无 |
| 训练开销 | 无额外参数 | 无额外参数 | $T_{\max} \times d$ 参数 |
| 因果掩码 | 天然兼容（仅看左侧） | 需显式 mask | 需显式 mask |

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import torch

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.position.alibi import ALiBiPositionalBias, get_alibi_slopes

### 2. 斜率生成验证

In [ ]:
for n_heads in [4, 8, 16]:
    slopes = get_alibi_slopes(n_heads)
    print(f"n_heads={n_heads:2d}:  slopes=[{", ".join(f"{s:.4f}" for s in slopes[:3])} ... {slopes[-1]:.4f}]")
    print(f"          最大/最小斜率比: {slopes[0] / slopes[-1]:.2f}")
    for i in range(1, n_heads):
        assert slopes[i] < slopes[i-1], "斜率应严格递减"
    assert slopes[0] > 0 and slopes[-1] > 0, "所有斜率应为正数"
    print()

### 3. 偏置矩阵形状与值域检查

In [ ]:
alibi = ALiBiPositionalBias(n_heads=8, max_seq_len=64)
bias = alibi(seq_len=32)

print("bias.shape:", tuple(bias.shape))   # (8, 32, 32)
print("bias[0, 0, :] 前5个值:", bias[0, 0, :5].tolist())
print("对角线值（i=j）:", bias[0].diagonal()[:5].tolist())
print("最远位置惩罚（i=0, j=31）:", bias[0, 0, 31].item())

assert bias.shape == (8, 32, 32), "形状应为 (n_heads, seq_len, seq_len)"
assert (bias.diagonal(dim1=-2, dim2=-1) == 0).all(), "对角线（i=j）应为 0"
assert bias[0, 0, -1] < 0, "远距离应有负偏置"

### 4. 偏置矩阵可视化

展示 ALiBi 偏置矩阵的热力图，观察不同头的局部性范围差异。

In [ ]:
import matplotlib.pyplot as plt

alibi = ALiBiPositionalBias(n_heads=8, max_seq_len=64)
bias = alibi(seq_len=32).numpy()  # (8, 32, 32)

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for idx, ax in enumerate(axes.flat):
    im = ax.imshow(bias[idx], cmap="viridis", aspect="auto")
    ax.set_title(f"Head {idx}  (slope={alibi.slopes[idx]:.4f})")
    ax.set_xlabel("Key position j")
    ax.set_ylabel("Query position i")
    fig.colorbar(im, ax=ax, fraction=0.046)

fig.suptitle("ALiBi Bias Matrix per Head (seq_len=32)", fontsize=14)
plt.tight_layout()
plt.show()

print("观察：斜率大的头（左侧）远距离惩罚更强，注意力更局部；")
print("      斜率小的头（右侧）允许更宽的注意力范围。")

### 5. 应用到 Attention Score 的演示

In [ ]:
torch.manual_seed(0)
batch, n_heads, seq_len = 2, 8, 16

attn_scores = torch.randn(batch, n_heads, seq_len, seq_len)

alibi = ALiBiPositionalBias(n_heads=n_heads, max_seq_len=seq_len)
modified = alibi.apply_to_attention(attn_scores)

print("原始 scores 均值:", attn_scores.mean().item())
print("ALiBi 后 scores 均值:", modified.mean().item())
print("scores 下降幅度:", (attn_scores - modified).mean().item())

diff = attn_scores - modified  # (B, H, T, T)
far_diff = diff[:, :, 0, 15].mean().item()
near_diff = diff[:, :, 5, 6].mean().item()
print(f"最远位置惩罚均值: {far_diff:.4f}")
print(f"最近位置惩罚均值: {near_diff:.4f}")
assert far_diff > near_diff, "远距离应受到更大惩罚"

### 6. 超长序列零成本外推验证

ALiBi 的核心优势：预训练的模型可直接处理更长的序列，无需任何修改。

In [ ]:
alibi = ALiBiPositionalBias(n_heads=4, max_seq_len=64)
print("初始缓存长度:", alibi._cached_seq_len)

bias_long = alibi(seq_len=200)
print("外推后长度:", alibi._cached_seq_len)
print("输出形状:", tuple(bias_long.shape))
assert bias_long.shape == (4, 200, 200), "应零成本外推到任意长度"

### 7. 源码精讲

```python
def get_alibi_slopes(n_heads: int) -> torch.Tensor:
    i = torch.arange(1, n_heads + 1, dtype=torch.float32)
    return 2 ** (-8.0 * i / n_heads)

class ALiBiPositionalBias(nn.Module):
    def __init__(self, n_heads: int, max_seq_len: int = 2048):
        super().__init__()
        self.n_heads = n_heads
        self.slopes = get_alibi_slopes(n_heads)  # (n_heads,)
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len: int):
        positions = torch.arange(seq_len, dtype=torch.float32)
        dist = positions.unsqueeze(0) - positions.unsqueeze(1)  # (T, T)
        dist = dist.abs()
        slopes = self.slopes.view(-1, 1, 1)
        bias = -(slopes * dist.unsqueeze(0))  # (H, T, T)
        self.register_buffer("bias_cached", bias, persistent=False)
```

关键设计点：
- 斜率通过简单的指数公式生成，无任何可学习参数。
- 偏置矩阵仅依赖相对距离 $|i-j|$，因此天然具备平移不变性和长度外推性。
- `apply_to_attention` 将偏置广播到 batch 维度并加到 attention logits 上，需在 causal mask 之前调用。
- 缓存动态扩展机制保证推理时任意长度都能即时响应。

---

## 延伸阅读与参考资料

### 核心论文
- **Train Short, Test Long: Attention with Linear Biases Enables Input Length Extrapolation**: Press et al., 2021. [arXiv:2108.12409](https://arxiv.org/abs/2108.12409)

### 工程实践
- **BLOOM**: BigScience 的 176B 多语言模型采用 ALiBi，在 2048 长度上训练后直接外推到更长序列。
- **MPT**: MosaicML 的 MPT-7B/30B 系列使用 ALiBi 实现高效的长上下文推理。

### 相关对比
- **RoPE + YaRN**: 在 RoPE 基础上做频率插值，需要额外计算；ALiBi 无需任何修改即可外推。
- **T5 Relative Bias**: 可学习的相对位置偏置，与 ALiBi 的固定线性偏置形成对比。